# Classwork: The Analyst's Toolkit

**Names:** _(pair)_

After learning about different tools (or a toolkit-- Descriptive statistics, Correlation, Regression/Multiple regression, Chi-square test) of working with data in the previous Mods, in this acitivity you will apply your real skills on the data you have.  
  
  Suppose your company hands you `customers.csv` (40 customers: `age`, `income`, `spending`, `region`, `subscribed`) and a list of questions. You are the analyst. Answer each one with the right tool/technique you decide to apply.

Work in pairs. Write answers in the markdown cells provided.

## Setup
Import the libraries you'll need and load the data.

In [30]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm

# load customers.csv into a DataFrame
df = pd.read_csv("./customers.csv", encoding="latin-1")
# take a first look (head, info)
df.head()
#df.info()

,customer_id,age,income,spending,region,subscribed
0,1,25,61.3,39.1,West,No
1,2,25,54.9,42.1,South,Yes
2,3,51,105.1,59.9,South,No
3,4,39,NaN,49.9,South,No
4,5,43,72.4,48.3,East,No


## Part 0: Know your data, then clean it

This data is dirty on purpose. Find and fix **three** problems before you analyze anything.

In [31]:
# HINT 1: a numeric column has missing values -> decide how to fill them (mean? median? why?)
# HINT 2: one category label is written in the wrong case -> standardize it
# HINT 3: one value is impossible for its column -> treat it as missing, then fill
# tip: check .isna().sum() and the unique values of the category column

def missing_values():
    print("Missing values per column:")
    null_values = df.isna().sum()
    print(null_values.to_string())

    print("\nPercentage of missing values:")
    pct_missing = null_values.sum()/(len(df) * len(df.columns))
    print(f"{pct_missing * 100:.2F}%")

    print("\nNumber of rows WITHOUT any missing values:")
    rows_with_missing = len((df[(df.income.isna()) | (df.spending.isna())]))
    print(len(df) - rows_with_missing)

missing_values()
#fixing values that are not possible
def fix_negative_values():
    df.loc[df['age'] <= 0, 'age'] = np.nan
    df.loc[df['income'] <= 0, 'income'] = np.nan
    df.loc[df['spending'] <= 0, 'spending'] = np.nan

fix_negative_values()

#replace income missing values with the mean
df["income"] = np.where(df["income"].isna(), df["income"].mean(), df["income"])


#fixing the weird
df["region"] = df["region"].replace(["east"], "East", regex=True)

missing_values()

Missing values per column:
customer_id    0
age            0
income         2
spending       0
region         0
subscribed     0

Percentage of missing values:
0.83%

Number of rows WITHOUT any missing values:
38
Missing values per column:
customer_id    0
age            0
income         0
spending       0
region         0
subscribed     0

Percentage of missing values:
0.00%

Number of rows WITHOUT any missing values:
40


**Explain each issue:** what you found and how you handled it.

*Your answer:* There were missing values in the income column, and negative values as well which is not possible. There was also messed up casing in the region column. I fixed the income issue by making the values equal to the mean of the income. I fixed the negative value by making the value null and then fixing it with the mean. Then I fixed the case issue by replacing the value.


## Part 1: Pick the tool for each question

For each question: (a) name the data types involved, (b) name the tool, (c) run it, (d) answer in one plain sentence.

### Q1. Do customers who earn more also spend more?

In [32]:
# data types: income = float, spending = float
# tool: pearson correlation coefficient
# hint: two continuous variables moving together
r, p = stats.pearsonr(df.income, df.spending)
print(f'r (Pearson correlation coefficient): {r:.3f}')
print(f'p-value:                             {p:.3f}')


r (Pearson correlation coefficient): 0.719
p-value:                             0.000


*Your answer:* Yes customers who earn more also spend more, the correlation is positive, meaning as one value increases the other value increases as well. The p-value is low meaning this is not a coincidence. 


### Q2. Is a customer's region related to whether they subscribed?

In [33]:
# data types: region = String, subscribed = Bool
# tool: contingency table
# hint: two categorical variables -> build a table with pd.crosstab, then test it
contingency_table = pd.crosstab(df['region'], df['subscribed'])
print(contingency_table)


subscribed  No  Yes
region             
East         5   16
South        6    3
West         9    1


*Your answer:*
Yes, the customer's region is related to whether they are subscribed, people are more likely to be subscribed if they are in the East, whereas in the other region it is less likely they will be subscribed. 

### Q3. Can we predict spending from income and age together? How much do they explain? (You can review your Assignment 4 for this)

In [34]:
# data types: predicting a continuous outcome from TWO predictors
# tool: ?
# hint: remember to add a constant; look at R-squared and each predictor's p-value
x = df[['income', 'age']]
y = df['spending']

x = sm.add_constant(x)

model = sm.OLS(y, x).fit()

predictions = model.predict(x)
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               spending   R-squared:                       0.522
Model:                            OLS   Adj. R-squared:                  0.496
Method:                 Least Squares   F-statistic:                     20.17
Date:                Mon, 05 Oct 2026   Prob (F-statistic):           1.19e-06
Time:                        16:09:57   Log-Likelihood:                -131.04
No. Observations:                  40   AIC:                             268.1
Df Residuals:                      37   BIC:                             273.1
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         19.0369      6.023      3.161      0.0

*Your answer:*
You can kind of predict spending from income and age but not very good because the r value is 0.522, which means only about 52% of values is predictable

### Q4. Which region has the most customers, and what is the average income overall?

In [35]:
# hint: count a categorical column
# take the mean of a numeric column
count = (df['region']).value_counts()
print(count)
mean_income = (df["income"].mean())
print("Income mean")
print(mean_income)

region
East     21
West     10
South     9
Name: count, dtype: int64
Income mean
86.6027027027027


*Your answer:* The region with the most customers is the East with 21 customers, and the income mean is 86.63 thousand.


## Part 2: The reveal

Answer these two in a sentence each (no code needed).

**a)** Look back at your four tools (Correlation, Chi-square test, Multiple regression, & Descriptive statistic)). What decided which tool you used each time? (Hint: it was not the wording of the question.)

*Your answer:* The thing that decided what we needed each time was the information we were seeking depends on the tool we used. If we needed to find how two things related we knew we would use correlation or a chi-square test.
 

**b)** In Q1 you found a relationship. Can you tell the company that higher income *causes* higher spending? Why or why not?

*Your answer:* No, because we found the correlation between the two but correlation does not equal causation.


## Bonus

In Q2 you ran a chi-square test. Look at its **expected counts** (not the observed counts). Do they all meet the assumption you learned in the chi-square module? What would you tell the company about how much to trust that result?

In [36]:
# hint: chi2_contingency returns the expected table as its 4th value
# check: are all expected counts >= 5?
chi2, p, dof, expected = stats.chi2_contingency(
    pd.crosstab(df['region'], df['subscribed']), correction=False)
contributions = (pd.crosstab(df['region'], df['subscribed']) - expected)**2 / expected
print(contributions.round(2))

subscribed    No   Yes
region                
East        2.88  2.88
South       0.50  0.50
West        3.20  3.20


*Your answer:*

### Push your code to GitHub